# 5 — Aerobic / Anaerobic Labelling (EndomondoHR)

**Purpose.** Assigns each running workout a binary label (`aerobik` / `anaerobik`) and a continuous score (`anaerobic_time_frac`) by computing the fraction of the workout spent above a personal heart-rate threshold. The threshold is derived from each person's own heart-rate range (floor + 0.70 x (ceiling - floor)), without clustering and without an age/HRmax formula. The label is later used in the GRU/LSTM training pipeline.

**Inputs**
- `data/endomondoHR_speed.csv` — processed EndomondoHR time series (speed filled and smoothed in the earlier steps)
- `functions.py` — helper functions (`parse_arr`, `run_zone_pipeline`)

**Outputs**
- `data/aerobik_anaerobik.csv` — per-workout label, continuous score and summary features
- `training_zone_histogram.png` — histogram of `anaerobic_time_frac` used to choose the cutoff

**Note on labels.** The class labels `aerobik` ("aerobic") and `anaerobik` ("anaerobic") are intentionally Turkish-named; they are part of the project's API contract and are kept unchanged in code and data. Likewise, the QC reason codes produced inside `functions.py` (`cok_kisa`, `bozuk_timestamp`, `hiz_cok_eksik`, `hr_cok_eksik`, `cok_uzun`) are kept as-is.

Part of RunSight - see the repo README.

# Aerobic / Anaerobic Classification — EndomondoHR

## Problem

We have ~70k running workouts from the EndomondoHR (FitRec) dataset.
Each workout consists of five time series sampled at 500 points:
position (latitude/longitude), altitude, speed and heart rate. Our goal is
to give each workout an **"aerobik"** (aerobic) or **"anaerobik"** (anaerobic)
label — i.e. whether the workout was done at a comfortable/sustainable pace
relative to the person's own capacity, or was a demanding effort above their
personal threshold. This label will later be added as a feature to the
training data of an LSTM model.

## Why we don't use plain clustering (K-Means)

Our first attempts (see `Kmeans.ipynb`) used K-Means and GMM clustering to
split workouts into 4 classes (easy/tempo/interval/long run). The result was
not useful: the clusters separated mostly by workout DURATION rather than by
workout TYPE — a 15-minute workout could be labelled "long run" because the
clustering algorithm gave too much weight to the "duration" feature. In short,
clustering could not discover the notion of "workout type" from the data on
its own.

## Our approach: direct computation

Instead of relying on clustering, we use a **direct threshold computation**
based on a well-known principle from sports science: Seiler's "polarized
training" model splits workouts roughly into two groups — time spent **BELOW**
the **personal heart-rate threshold** (aerobic, sustainable) and time spent
**ABOVE** it (anaerobic, demanding). We do not use age or a maximum heart rate
formula (such as 220 minus age) because the dataset has no age information;
instead, we take the heart-rate range observed in each person's OWN past
workouts (lowest to highest) as the reference and set the "threshold" at 70%
of that range. We measure how much of a workout was spent above this
threshold and label it accordingly.

All functions used in this notebook are defined in `functions.py` (it must be
in the same folder). The code contains no machine-learning model selection such
as clustering, K selection or BIC — every step is a direct, explainable
computation.

## 0. Setup

We import the required libraries and the helper functions from `functions.py`
(under the alias `fc`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import functions as fc



pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Load the data

`endomondoHR_speed.csv` is the processed version of the raw EndomondoHR data —
missing `speed` values were computed from the GPS coordinates (distance via the
haversine formula, divided by time) and smoothed with a Savitzky-Golay filter
(see `haversine_savgol_filter.ipynb`).

The `longitude`, `altitude`, `latitude`, `heart_rate`, `timestamp` and `speed`
columns in the CSV are actually time series, but they are stored as plain text
(strings) in the CSV — each cell looks like `"[52.22, 52.23, ...]"`.
`fc.parse_arr` converts this string into a real numeric array (numpy array);
without this conversion no numeric operation (mean, difference, etc.) is
possible.

In [ ]:
CSV_PATH = "data/endomondoHR_speed.csv"

df = pd.read_csv(CSV_PATH)

In [ ]:
num_cols = ['longitude', 'altitude', 'latitude', 'heart_rate', 'timestamp', 'speed']

for col in num_cols:
    df[col] = df[col].apply(fc.parse_arr)

### Verify that the arrays were really parsed

Parsing (converting a string into a numeric array) can fail silently: given a
malformed string, `parse_arr` does not raise an error, it returns an empty
array (`[]`). This is not noticeable at first glance — but all later
computations (mean speed, heart-rate range, etc.) produce meaningless or wrong
results for that row, and sometimes crash silently. So after loading we always
run a small check: are the array lengths of all rows as expected (here, 500),
and are the duration/speed/heart-rate values of a random row realistic?

In [ ]:
lens = df["speed"].apply(lambda a: len(a) if isinstance(a, np.ndarray) else -1)
print("array lengths:", lens.value_counts().head().to_dict())

row = df.iloc[0]
print("first record — duration (min):", round((row['timestamp'][-1] - row['timestamp'][0]) / 60, 1))
print("speed range:", round(np.nanmin(row['speed']), 1), "-", round(np.nanmax(row['speed']), 1), "km/h")
print("heart-rate range:", round(np.nanmin(row['heart_rate'])), "-", round(np.nanmax(row['heart_rate'])), "bpm")

## 2. Run the pipeline

`fc.run_zone_pipeline` runs all steps from the raw data to the final label in a
single call. Internally, the following happens in order:

1. **Quality control (QC)** — each workout is evaluated individually.
   Workouts that are too short (<10 min) or too long (>300 min), or in which
   more than 20% of the speed values or more than half of the heart-rate values
   are physically impossible (e.g. a running speed of 300 km/h, a heart rate of
   400 bpm — probably sensor errors), are dropped entirely. Isolated bad points
   within the limits are not deleted but estimated linearly from neighbouring
   points (interpolation) and filled in — so the rest of the workout is not
   thrown away.
2. **Feature extraction** — numeric features such as duration, distance and
   speed statistics (median, variability) are extracted from each clean
   workout.
3. **Personal normalization** — "11 km/h" may be an easy pace for one person and
   a race pace for another. So each person's speed/heart-rate values are
   divided by their OWN past median (e.g. "this workout was 1.2 times faster
   than this person's norm").
4. **Heart-rate zone threshold** — between the lowest and highest heart-rate
   values observed across ALL of a person's workouts, the point corresponding to
   70% of the range is set as the "personal threshold" (the VT1 / aerobic
   threshold approach from sports science). This is a reference derived
   entirely from the person's own data, without an age/HRmax formula.
5. **Labelling** — the fraction of a workout's duration spent above this
   threshold is recorded as `anaerobic_time_frac`; if this fraction exceeds a
   certain cutoff value (`cutoff`) the workout is labelled "anaerobik",
   otherwise "aerobik".

There is no clustering (K-Means/GMM), K selection or model comparison (such as
BIC) in any step — everything is a direct, step-by-step traceable computation.

In [ ]:
X, art = fc.run_zone_pipeline(df, cutoff=0.02)
X[["training_zone", "anaerobic_time_frac", "duration_min", "hr_reserve_frac"]].head(10)

## 3. Look at the distribution

For the "anaerobik" decision on `anaerobic_time_frac` (the fraction of a
workout spent above the threshold), instead of picking the threshold
(`cutoff`) arbitrarily, we determine it by looking at the distribution of the
data itself. The expected shape is as follows: the vast majority of values
should pile up very close to zero (these are workouts done at an aerobic,
sustainable pace), then there should be a clear gap, followed by a separate
"tail" (these are genuinely demanding, above-threshold workouts). We put the
`cutoff` value at the point where this gap ends — the histogram lets us see
this by eye.

In [ ]:
def training_zone_histogram(X, bins=40, path="../docs/images/training_zone_histogram.png"):
    """
    Distribution of anaerobic_time_frac — choose the cutoff by looking at the
    distribution, not by eyeballing. A large pile-up at zero plus a separate
    tail is expected; put the cutoff where the pile-up ends. The y axis is
    log-scale so that the pile-up and the tail are visible at the same time.

    NOTE: there is deliberately NO `matplotlib.use("Agg")` here. It used to be
    here and had a serious side effect: it changes the backend GLOBALLY — not
    only inside this function but for the rest of the notebook. In
    Colab/Jupyter, no `plt.show()` after this function ran (e.g. `show_samples`
    in the visual verification section) displayed anything — silently, without
    an error. If a headless test environment (server, no display) is needed, set
    it at the top of the calling script (OUTSIDE the function), do not embed it
    in a library function.
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(X["anaerobic_time_frac"], bins=bins, color="steelblue", edgecolor="white")
    ax.set_xlabel("anaerobic_time_frac (fraction of time spent above the personal threshold)")
    ax.set_title("Choose the aerobic/anaerobic cutoff here")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=110)
    plt.close(fig)
    return path

In [ ]:
print(X["training_zone"].value_counts())
print(f"\nanaerobic share: %{100*(X['training_zone']=='anaerobik').mean():.1f}")

path = training_zone_histogram(X)
from IPython.display import Image, display
display(Image(path))

### What to look for when evaluating the cutoff choice

**Good sign:** a large pile-up very close to zero, then a clear gap. This shows
that aerobic and anaerobic workouts really form two separate groups — putting
the `cutoff` inside this gap is a reasonable decision.

**Bad/ambiguous sign:** no gap, the distribution rises smoothly from zero. In
this case no `cutoff` value is "correct" — using the continuous score
(`anaerobic_time_frac`) directly instead of the binary (aerobic/anaerobic)
label would be a more honest approach.

**A calibration issue to watch for:** a pile-up in the tail of the distribution
(between 0.85 and 1.0, i.e. workouts in which ALMOST THE WHOLE workout was
spent above the threshold) is suspicious. Physiologically, sustaining almost an
entire 60+ minute workout above the threshold is very hard (the body tires, the
pace drops). If we see such a pile-up, there are two likely explanations: (1)
that person has very few workouts and fell back to a general (non-personal)
threshold, or (2) that person's "personal heart-rate ceiling" was computed from
a single, corrupted workout and came out much lower than the true value — which
makes all their other workouts look artificially "far above the threshold". The
cell below tests these two possibilities.

In [ ]:
# Test the source of the suspiciously high anaerobic_time_frac values
counts = X.groupby("_user").size()
few = counts[counts < 5].index
print(f"Users with fewer than 5 workouts: {len(few)} / {counts.shape[0]}")
print(f"Workouts belonging to these users: {X['_user'].isin(few).sum()} / {len(X)}")

high = X["anaerobic_time_frac"] > 0.8
print(f"\nWorkouts with anaerobic_time_frac > 0.8: {high.sum()}")
print(f"  share of these from 'users with few workouts': "
      f"%{100*X.loc[high, '_user'].isin(few).mean():.1f}")
print(f"  share in the overall dataset (for comparison): "
      f"%{100*X['_user'].isin(few).mean():.1f}")

# If the two shares above are close to each other, the cause is NOT "few workouts" —
# then check whether a single corrupted workout distorts the personal ceiling:
susp_users = X.loc[high, "_user"].unique()
print(f"\nFor the {len(susp_users)} users with suspicious workouts, the hr_p95 range "
      f"within each user (wide range = a single workout may be distorting it):")
print(X[X['_user'].isin(susp_users)].groupby('_user')['hr_p95'].agg(['min','max','count']).head(10))

## 4. Visual verification

Numeric statistics (histogram, ratios) can be misleading; the most reliable way
to make sure the label is actually meaningful is to personally look at the raw
speed and heart-rate plots of a few workouts. The visual signature we expect is
this: in samples labelled **"anaerobik"**, the heart-rate plot should show a
clear, distinct section above the personal threshold (heart rate rises as the
effort increases and stays there). In samples labelled **"aerobik"**, the heart
rate should stay steadier and lower, without sharp jumps. If in a few samples
you see the exact opposite of the expected pattern (e.g. labelled "anaerobik"
but with a flat heart-rate line), it indicates a bug in the pipeline — numeric
metrics can hide that bug but a plot cannot.

In [ ]:
def show_samples(X, df, mask, title, n=3, seed=0):
    pool = X[mask]
    if len(pool) == 0:
        print(f"{title}: no samples"); return
    sel = pool.sample(min(n, len(pool)), random_state=seed)
    fig, axes = plt.subplots(2, len(sel), figsize=(4.3 * len(sel), 5.2), squeeze=False)
    for col, (idx, r) in enumerate(sel.iterrows()):
        axes[0, col].plot(df.loc[idx, "speed"], lw=0.7)
        axes[0, col].set_title(f"{title} — speed\nfrac={r.anaerobic_time_frac:.3f}", fontsize=9)
        axes[0, col].grid(alpha=0.25)
        axes[1, col].plot(df.loc[idx, "heart_rate"], lw=0.7, color="crimson")
        axes[1, col].set_title("heart rate", fontsize=9)
        axes[1, col].grid(alpha=0.25)
    plt.tight_layout(); plt.show()

show_samples(X, df, X["training_zone"] == "anaerobik", "ANAEROBIC", n=3)
show_samples(X, df, X["training_zone"] == "aerobik", "AEROBIC", n=3)

## 5. Save the result

We write the final label for each workout (`training_zone`) and the continuous
score (`anaerobic_time_frac`) — plus duration, distance, speed and heart-rate
summaries for interpretability — to `data/aerobik_anaerobik.csv`. This file will
be used as input in the next step of the project (adding this label as a feature
to the LSTM's training data).

In [ ]:
out_cols = ["training_zone", "anaerobic_time_frac", "duration_min",
           "distance_km", "speed_med", "hr_med", "hr_reserve_frac"]
out = X[out_cols]
os.makedirs("data", exist_ok=True)
out.to_csv("data/aerobik_anaerobik.csv", index=True)
print(f"{len(out)} workouts saved -> data/aerobik_anaerobik.csv")
out["training_zone"].value_counts()